# Sentiment Classification Project

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Verify Setup
Make sure cuda (GPU) is available

# Load data

In [2]:
train_full = pd.read_csv("data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [3]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [4]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [5]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [6]:
from embeddings import *
from models import *
from train_loop_caching import train_loop
from embeddings_adv import *


/Users/taekim/cil_project/Garnella/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Cell 2: Model comparison (best embeddings, different classifiers)
# combinations_models = [
#     (get_gemma_embeddings, get_logistic_regression),
#     (get_gemma_embeddings, get_linear_svm),
#     (get_gemma_embeddings, get_knn),
#     (get_gemma_embeddings, get_mlp),
#     # (get_gemma_embeddings, get_xgboost),
# ]
# results_models = train_loop(train_df, val_df, combinations_models)
# pd.DataFrame(results_models).sort_values("validation_score", ascending=False)

# Gemma Embeddings — Baseline Classifier Results
Embeddings: `get_gemma_embeddings` (EmbeddingGemma-300m, `Classification` prompt)

### Hyperparameters
| Classifier | Hyperparameters |
|---|---|
| LogisticRegression | `C=1.0, max_iter=1000` |
| LinearSVC | `C=1.0, max_iter=1000` |
| KNeighborsClassifier | `n_neighbors=15, n_jobs=-1` |
| MLPClassifier | `hidden_layer_sizes=(256,), max_iter=300, random_state=1` |
| MLPClassifier-v2 | `hidden_layer_sizes=(128,), alpha=1e-2, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, max_iter=300, random_state=1` |
| RandomForestClassifier | `n_estimators=300, random_state=1` |

## Results
| # | Classifier | Train Score | Train MAE | Train Acc | **Val Score** | **Val MAE** | **Val Acc** |
|---|---|---|---|---|---|---|---|
| 4 | MLPClassifier-v2 | 0.9008 | 0.3968 | 0.6547 | **0.8950** ⬅️ | **0.4201** | **0.6342** ⬅️ |
| 0 | LogisticRegression | 0.8936 | 0.4256 | 0.6321 | **0.8917** | **0.4331** | **0.6256** |
| 5 | RandomForestClassifier | 1.0000 | 0.0000 | 1.0000 | **0.8866** | **0.4537** | **0.6062** |
| 1 | LinearSVC | 0.8877 | 0.4493 | 0.6225 | **0.8852** | **0.4593** | **0.6145** |
| 2 | KNeighborsClassifier | 0.8917 | 0.4334 | 0.6425 | **0.8747** | **0.5012** | **0.5796** |
| 3 | MLPClassifier | 0.9947 | 0.0214 | 0.9819 | **0.8593** | **0.5628** | **0.5435** |

In [ ]:
combinations_adv = [
    (get_gemma_embeddings, get_random_forest_v2),
    (get_gemma_embeddings_v2,          get_random_forest_v2),
    (get_gemma_embeddings_seq128,          get_random_forest_v2),
    (get_gemma_embeddings_v2,          get_logistic_regression),
    (get_gemma_embeddings_v2,          get_linear_svm),
    (get_gemma_embeddings_v2,          get_mlp),
    (get_gemma_embeddings_v2,          get_knn),
    (get_gemma_embeddings_seq128,          get_logistic_regression),
    (get_gemma_embeddings_seq128,          get_linear_svm),
    (get_gemma_embeddings_seq128,          get_mlp),
    (get_gemma_embeddings_seq128,          get_knn),
]
results_local = train_loop(train_df, val_df, combinations_adv)
pd.DataFrame(results_local).sort_values("validation_score", ascending=False)

<>:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
/var/folders/ry/qhyv3_sd28n3rzthk9tzgl5r0000gn/T/ipykernel_48854/150557603.py:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  (get_gemma_embeddings_seq128,          get_random_forest_v2)


TypeError: 'tuple' object is not callable

In [9]:
combinations_rf = [
    (get_gemma_embeddings, get_random_forest_v2),
    (get_gemma_embeddings_v2,          get_random_forest_v2),
    (get_gemma_embeddings_seq128,          get_random_forest_v2)
]
results_models = train_loop(train_df, val_df, combinations_rf)
pd.DataFrame(results_models).sort_values("validation_score", ascending=False)

Loading saved embeddings from ./embedding_cache
Done with embeddings: get_gemma_embeddings
Combination: get_gemma_embeddings + RandomForestClassifier
Training Score: 0.8994, MAE: 0.4025, Accuracy: 0.6567
Validation Score: 0.8834, MAE: 0.4666, Accuracy: 0.5963
Loading cached gemma_v2 embeddings
Done with embeddings: get_gemma_embeddings_v2


KeyboardInterrupt: 